# Introduction to Recommender Systems

<p align="center">
    <img width="721" alt="cover-image" src="https://user-images.githubusercontent.com/49638680/204351915-373011d3-75ac-4e21-a6df-99cd1c552f2c.png">
</p>

---

# 🎓 Content-Based Filtering

---

## Introduction

A detailed exploration of CBF, covering theory, equations, metrics, and ML-based training.

# 📌 Content-Based Filtering (CBF): A Complete Guide
*A detailed exploration of CBF, covering theory, equations, metrics, and ML-based training.*

---

## **1️⃣ Introduction: What is Content-Based Filtering?**
Content-Based Filtering (**CBF**) is a **recommendation technique** that suggests items to users **based on item features** rather than other users' interactions. It is widely used in **search engines, personalized content delivery, and document recommendations**.

### **🔹 How It Works**
- **Each item (e.g., document, movie, product)** is represented by a **feature vector**.
- The system **compares item features** to those of items a user has interacted with.
- The system recommends **items similar to those the user liked in the past**.

### **🔹 Key Characteristics**
✔ **Does not rely on other users' preferences** (unlike collaborative filtering).  
✔ **Requires metadata or content features** (e.g., text, keywords, embeddings).  
✔ **Great for personalised recommendations** but **suffers from over-specialization**.

### **🔹 Real-World Applications**
✔ **Search engines**: Ranking documents based on query similarity.  
✔ **News recommendation**: Suggesting articles similar to ones a user has read.  
✔ **E-learning**: Recommending study materials similar to previous lessons.  

---

## **2️⃣ Mathematical Foundations of Content-Based Filtering**
CBF operates by representing items as **feature vectors** and measuring their **similarity**.

### **🔹 Step 1: Representing Items as Feature Vectors**
Let **N** items be represented as **d-dimensional vectors**:

$$
\mathbf{x}_i = (x_{i1}, x_{i2}, ..., x_{id}) \in \mathbb{R}^d
$$

where:
- $ x_i $ represents the **i-th document/item**.
- Each element $ x_{ij} $ represents a **feature weight** (e.g., word frequency in TF-IDF).

### **🔹 Step 2: Computing Similarity Between Items**
The similarity between two items is calculated using **Cosine Similarity**:

$$
\text{sim}(\mathbf{x}_i, \mathbf{x}_j) = \frac{\mathbf{x}_i \cdot \mathbf{x}_j}{\|\mathbf{x}_i\| \|\mathbf{x}_j\|}
$$

where:
- $ \mathbf{x}_i \cdot \mathbf{x}_j $ is the **dot product**.
- $ \|\mathbf{x}_i\| $ and $ \|\mathbf{x}_j\| $ are the **vector magnitudes**.

If **sim(x1, x2) = 1**, they are identical; if **sim(x1, x2) = 0**, they are unrelated.

---

## **3️⃣ Implementing a Basic Content-Based Recommender System**

We first **vectorise documents using TF-IDF** and then **compute similarity scores**.

### **🔹 Step 1: Feature Extraction (TF-IDF)**
```python
from sklearn.feature_extraction.text import TfidfVectoriser

# Sample documents (text corpus)
documents = [
    "Deep learning revolutionises AI.",
    "AI is transforming industries.",
    "Bioinformatics integrates biology and data science.",
    "Natural language processing models are complex."
]

# Convert documents into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words='english')
doc_vectors = vectorizer.fit_transform(documents).toarray()
```

### **🔹 Step 2: Compute Similarity Scores**

```python
from sklearn.metrics.pairwise import cosine_similarity

# Compute similarity matrix
similarity_matrix = cosine_similarity(doc_vectors, doc_vectors)

# Print similarity scores
import pandas as pd
df_sim = pd.DataFrame(similarity_matrix, index=documents, columns=documents)
print(df_sim)
```

---

## **4️⃣ Evaluating the Recommender System**

Once recommendations are generated, we must evaluate their effectiveness. We have seen the several kinds of metrics used to evaluate recommender systems.

### 🔹 Precision@K and Recall@K

$$
\text{Precision@K} = \frac{\text{number of relevant recommended items in top-K}}{K}
$$

$$
\text{Recall@K} = \frac{\text{number of relevant recommended items in top-K}}{\text{total number of relevant items}}
$$

### 🔹 Normalised discount cumulative gain (NDCG)

$$
\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}
$$

where, 

$$
\text{DCG@K} = \sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i+1)}
$$
where, $rel_i$ is the ground truth relevance score of the $i^{th}$ item in the top-K list.

---

## **5️⃣ Train a ML model for content-based filtering**

Instead of using cosine similarity as it is, we now train an ML model to predict item relevance.

### 🔹 Step 1: Get user interaction data

The first step is preparing the data for training. We will use the user-item interaction data for this purpose.
These are contained in the dataframe `df_ranking` we imported a couple of lectures ago.

### 🔹 Step 2: Train a model

We will use a simple Logistic Regression model for this purpose. We will use the `LogisticRegression` class from the `sklearn.linear_model` module.

### 🔹 Step 3: Evaluate the model

We can make use of the metrics defined above to assess the performance of our model.

---

## **6️⃣ Learning-to-Rank: Optimising the Ranking Order**

A more advanced and interesting approach is Learning-to-Rank (LTR), which directly optimizes ranking performance.

### **🔹 Listwise Learning-to-Rank with XGBoost**

XGBoost is a popular gradient boosting library that can be used for LTR. It is a powerful tool for regression and classification tasks, but it can also be used for ranking tasks.

```python
from xgboost import XGBRanker

# Define group structure (how many docs per query)
group = [len(df_interactions)]

# Train an XGBoost Ranker
ltr_model = XGBRanker(
    objective="rank:ndcg",
    eval_metric="ndcg",
    booster="gbtree",
    eta=0.1,
    max_depth=3,
    n_estimators=50
)

ltr_model.fit(X, y, group)
```


We can use this model to predict and rank the movies for a given user.

```python
# Predict relevance scores
predicted_scores = ltr_model.predict(X)

# Rank documents based on predicted scores
df_interactions["predicted_relevance"] = predicted_scores
df_interactions.sort_values("predicted_relevance", ascending=False, inplace=True)

print(df_interactions[["doc_id", "predicted_relevance"]])
```
---

## ** 7️⃣ Summary: Comparing Methods **

| **Method** | **Pros** | **Cons** |
|-----------|---------|---------|
| **Cosine Similarity** | Simple, No Training Required | Cannot learn from user behaviour |
| **Logistic Regression** | Learns from data | May not optimise ranking well |
| **XGBoost Ranker (LTR)** | Optimised for ranking tasks | Requires labelled training data |

🚀 **Next Steps:**  
1. **Expand to large datasets** (e.g., Wikipedia, ArXiv).  
2. **Use deep learning models (BERT Ranker)**.  
3. **Integrate hybrid filtering (content + collaborative filtering)**. 


##  Packages <img align="left" src="../images/film_strip_vertical.png"     style=" width:40px;   " >
We will use the now familiar NumPy and pandas Packages.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

import random

import matplotlib.pyplot as plt
import seaborn as sns

# set plot size
plt.rcParams["figure.figsize"] = (20, 13)
sns.set_theme()
%matplotlib inline
%config InlineBackend.figure_format = "retina"

## 📚 Guided Exercise: Implementing Content-Based Filtering on a Books Recommendation Dataset

The aim of this exercise is to implement a content-based filtering system for a books recommendation dataset. The dataset contains information about books, including their titles, authors, genres, and descriptions. The goal is to recommend books to users based on their preferences and the content of the books.

### Step 1: Load the Dataset

Load the dataset into a pandas DataFrame. The dataset is available at [this link](https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset).

There are 3 interesting files in the dataset: 

* books.csv: contains information about the books.
* users.csv: contains information about the users.
* ratings.csv: contains information about the ratings given by the users to the books.

### Step 2: Preprocess the Data

Preprocess the data to make it suitable for content-based filtering. This should include:

* Handling missing values
* Encoding categorical variables
* Creating a content-based feature matrix

### Step 3: Implement Content-Based Filtering

Implement content-based filtering to recommend books to users. This should include:

* Calculating the similarity between books (you can use the system you prefer)
* Recommending books to users based on their preferences

### Step 4: Evaluate the System

Evaluate the performance of the content-based filtering system. This should include:

* Calculating metrics such as precision, recall, and F1-score
* Comparing the performance of the content-based filtering system with at least one other recommendation system.

### Step 5: Visualise the Results

Visualize the results of the content-based filtering system. This should include:

* Creating visualisations to show the performance of the system
* Creating visualisations to show the recommendations made by the system
